# Feature Engineering — installments_payments.csv

**¿Qué es esta tabla?**
Contiene el historial de pagos de cuotas de los créditos anteriores en Home Credit.
Cada fila es un pago (o cuota pendiente) de un crédito anterior de un cliente.
Con 13.6M de filas es la segunda tabla más grande.

**¿Por qué es valiosa?**
Es la tabla más directa sobre comportamiento de pago: cuánto debía pagar,
cuánto pagó realmente, y cuántos días tarde lo hizo. Un cliente que siempre
paga tarde o paga menos de lo que debe es una señal de alarma clara.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/raw/'
print('Librerías cargadas.')

Librerías cargadas.


## 1. Exploración

In [2]:
inst = pd.read_csv(DATA_PATH + 'installments_payments.csv')
print(f'Shape: {inst.shape}')
print(f'Columnas: {inst.columns.tolist()}')
print(f'Clientes únicos: {inst["SK_ID_CURR"].nunique():,}')
print(f'Pagos por cliente (promedio): {len(inst) / inst["SK_ID_CURR"].nunique():.1f}')
print(f'\nMissings:')
print(inst.isnull().mean().round(3)[inst.isnull().mean() > 0])

Shape: (13605401, 8)
Columnas: ['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']
Clientes únicos: 339,587
Pagos por cliente (promedio): 40.1

Missings:
DAYS_ENTRY_PAYMENT    0.0
AMT_PAYMENT           0.0
dtype: float64


## 2. Feature Engineering

Las features más poderosas de esta tabla vienen de comparar lo que debía pagar
con lo que pagó realmente, y cuántos días tarde lo hizo.

In [3]:
# Features derivadas a nivel de cuota — comparar deber vs pagar
inst['INST_DAYS_LATE'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['INST_AMT_DIFF'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
inst['INST_PAYMENT_RATIO'] = inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1)

# Flags binarios por cuota
inst['INST_PAID_LATE'] = (inst['INST_DAYS_LATE'] > 0).astype(int)
inst['INST_PAID_SHORT'] = (inst['INST_AMT_DIFF'] > 0).astype(int)

inst_agg = inst.groupby('SK_ID_CURR').agg(
    # Conteos
    INST_COUNT=('SK_ID_PREV', 'count'),
    INST_PAID_LATE_COUNT=('INST_PAID_LATE', 'sum'),
    INST_PAID_SHORT_COUNT=('INST_PAID_SHORT', 'sum'),

    # Días de retraso
    INST_DAYS_LATE_MEAN=('INST_DAYS_LATE', 'mean'),
    INST_DAYS_LATE_MAX=('INST_DAYS_LATE', 'max'),
    INST_DAYS_LATE_STD=('INST_DAYS_LATE', 'std'),

    # Diferencia de monto
    INST_AMT_DIFF_MEAN=('INST_AMT_DIFF', 'mean'),
    INST_AMT_DIFF_MAX=('INST_AMT_DIFF', 'max'),
    INST_AMT_DIFF_SUM=('INST_AMT_DIFF', 'sum'),

    # Ratio de pago
    INST_PAYMENT_RATIO_MEAN=('INST_PAYMENT_RATIO', 'mean'),
    INST_PAYMENT_RATIO_MIN=('INST_PAYMENT_RATIO', 'min'),

    # Montos absolutos
    INST_AMT_INSTALMENT_SUM=('AMT_INSTALMENT', 'sum'),
    INST_AMT_PAYMENT_SUM=('AMT_PAYMENT', 'sum'),
).reset_index()

# Ratios finales por cliente
inst_agg['INST_LATE_RATIO'] = inst_agg['INST_PAID_LATE_COUNT'] / (inst_agg['INST_COUNT'] + 1)
inst_agg['INST_SHORT_RATIO'] = inst_agg['INST_PAID_SHORT_COUNT'] / (inst_agg['INST_COUNT'] + 1)

print(f'Shape inst_agg: {inst_agg.shape}')
print(f'Features generadas: {inst_agg.shape[1] - 1}')
inst_agg.head(3)

Shape inst_agg: (339587, 16)
Features generadas: 15


,SK_ID_CURR,INST_COUNT,INST_PAID_LATE_COUNT,INST_PAID_SHORT_COUNT,INST_DAYS_LATE_MEAN,INST_DAYS_LATE_MAX,INST_DAYS_LATE_STD,INST_AMT_DIFF_MEAN,INST_AMT_DIFF_MAX,INST_AMT_DIFF_SUM,INST_PAYMENT_RATIO_MEAN,INST_PAYMENT_RATIO_MIN,INST_AMT_INSTALMENT_SUM,INST_AMT_PAYMENT_SUM,INST_LATE_RATIO,INST_SHORT_RATIO
0,100001,7,1,0,-7.285714,11.0,14.625483,0.0,0.0,0.0,0.999776,0.999747,41195.925,41195.925,0.125,0.0
1,100002,19,0,0,-20.421053,-12.0,4.925171,0.0,0.0,0.0,0.999897,0.999892,219625.695,219625.695,0.000,0.0
2,100003,25,0,0,-7.160000,-1.0,3.726929,0.0,0.0,0.0,0.999922,0.999850,1618864.650,1618864.650,0.000,0.0


## 3. Guardar y validar

In [4]:
inst_agg.to_parquet('../data/processed/installments_features.parquet', index=False)
print('Guardado en data/processed/installments_features.parquet')

app = pd.read_csv(DATA_PATH + 'application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
val = app.merge(inst_agg, on='SK_ID_CURR', how='left')

print(f'\nClientes SIN historial en installments: {val["INST_COUNT"].isna().sum():,} ({val["INST_COUNT"].isna().mean()*100:.1f}%)')

print('\nCorrelación con TARGET (top features):')
corr = val.drop(columns='SK_ID_CURR').corr()['TARGET'].drop('TARGET')
print(corr.abs().sort_values(ascending=False).head(10).round(4))

Guardado en data/processed/installments_features.parquet

Clientes SIN historial en installments: 15,868 (5.2%)

Correlación con TARGET (top features):
INST_LATE_RATIO            0.0692
INST_SHORT_RATIO           0.0617
INST_PAID_LATE_COUNT       0.0304
INST_AMT_DIFF_MEAN         0.0293
INST_PAID_SHORT_COUNT      0.0283
INST_AMT_DIFF_SUM          0.0273
INST_AMT_PAYMENT_SUM       0.0244
INST_COUNT                 0.0211
INST_DAYS_LATE_MEAN        0.0209
INST_AMT_INSTALMENT_SUM    0.0198
Name: TARGET, dtype: float64
